# FIT4112 · Lab 1 · Notebook 01 — Làm sạch dữ liệu và EDA

**Bài toán:** Dữ liệu bán lẻ đang chứa giao dịch không hợp lệ. Ta cần dữ liệu sạch trước khi phân tích giỏ hàng.

**Bạn sẽ học:** đọc CSV, kiểm tra thiếu dữ liệu, loại hóa đơn huỷ, lọc số lượng/đơn giá, xem thống kê đơn giản.

> Đây là lab dành cho người học; hãy chạy tuần tự các cell. Không cần Papermill. Dữ liệu mặc định là dữ liệu giả lập để thực hành.

## 1. Thiết lập

**Mặc định:** dữ liệu demo nhỏ chạy nhanh. **Mở rộng:** đặt `online_retail.csv` vào `data/raw` rồi đổi tên `DATA_FILE`.

In [ ]:
from pathlib import Path
import pandas as pd

def find_project_root():
    """Tìm thư mục gốc dù notebook chạy từ root hay notebooks/."""
    cwd = Path.cwd().resolve()
    for folder in [cwd, *cwd.parents]:
        if (folder / "data" / "raw" / "transactions_demo.csv").exists():
            return folder
    raise FileNotFoundError("Không tìm thấy data/raw/transactions_demo.csv. Hãy clone đúng repo.")

ROOT = find_project_root()
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

DATA_FILE = "transactions_demo.csv"  # tùy chọn: "online_retail.csv"
COUNTRY = "United Kingdom"

## 2. Đọc và xem dữ liệu

In [ ]:
df = pd.read_csv(
    RAW_DIR / DATA_FILE,
    encoding="ISO-8859-1",
    dtype={"InvoiceNo": "string", "StockCode": "string", "CustomerID": "string"},
)
print("Số dòng ban đầu:", len(df))
print("Số hóa đơn:", df["InvoiceNo"].nunique())
display(df.head())
display(df.isna().sum().to_frame("Số giá trị thiếu"))

## 3. Làm sạch có giải thích

- Hóa đơn bắt đầu bằng `C`: giao dịch bị hủy.
- Số lượng và đơn giá phải dương.
- Thiếu tên sản phẩm/hóa đơn: không thể tạo giỏ hàng.
- Chỉ phân tích thị trường `United Kingdom` để các lần chạy có cùng phạm vi.

**Câu hỏi:** Nếu không lọc hóa đơn hủy, basket có thể bị ảnh hưởng thế nào?

In [ ]:
df_clean = df.copy()
df_clean["Quantity"] = pd.to_numeric(df_clean["Quantity"], errors="coerce")
df_clean["UnitPrice"] = pd.to_numeric(df_clean["UnitPrice"], errors="coerce")
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"], errors="coerce")
df_clean["Description"] = df_clean["Description"].astype("string").str.strip()

valid = (
    df_clean["InvoiceNo"].notna()
    & ~df_clean["InvoiceNo"].str.startswith("C", na=False)
    & df_clean["Description"].notna()
    & df_clean["Description"].ne("")
    & df_clean["Quantity"].gt(0)
    & df_clean["UnitPrice"].gt(0)
    & df_clean["Country"].eq(COUNTRY)
)
df_clean = df_clean.loc[valid].copy()
df_clean["TotalPrice"] = df_clean["Quantity"] * df_clean["UnitPrice"]

print(f"Ban đầu: {len(df):,} dòng")
print(f"Giữ lại: {len(df_clean):,} dòng")
print(f"Loại bỏ: {len(df)-len(df_clean):,} dòng")
print(f"Số hóa đơn còn lại: {df_clean['InvoiceNo'].nunique():,}")
display(df_clean.head())

## 4. Khám phá ngắn: sản phẩm và doanh thu

In [ ]:
top_products = df_clean.groupby("Description")["Quantity"].sum().sort_values(ascending=False).head(10)
display(top_products.rename("Số lượng bán").to_frame())
ax = top_products.sort_values().plot.barh(title="10 sản phẩm bán nhiều nhất", figsize=(8, 4))
ax.set_xlabel("Tổng số lượng")
print("Tổng doanh thu (£):", round(df_clean["TotalPrice"].sum(), 2))

## 5. Lưu kết quả để notebook 02 đọc

In [ ]:
output_path = OUT_DIR / "cleaned_transactions.csv"
df_clean.to_csv(output_path, index=False)
assert output_path.exists()
print("Đã lưu:", output_path)

## Tự kiểm tra

1. Có bao nhiêu dòng bị loại?
2. Tại sao sử dụng `InvoiceNo` để nhóm giao dịch, không phải `CustomerID`?
3. Trong tập dữ liệu thật, có tình huống nào cần làm sạch thêm không?

**Tiếp theo:** mở `02_basket_preparation.ipynb`.